In [0]:
from datetime import datetime, timedelta

today = datetime.utcnow().strftime("%Y-%m-%d")
print(f"[INFO] 집계 기준일: {today}")

In [0]:
# ── 셀 1: 환경설정 ────────────────────────────────────
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()

# Databricks Secret Scope를 통한 ADLS OAuth 설정
SCOPE = "kv-sense-team4"
ACCOUNT = "datacopsadls"

client_id     = dbutils.secrets.get(SCOPE, "adls-client-id")
client_secret = dbutils.secrets.get(SCOPE, "adls-client-secret")
tenant_id     = dbutils.secrets.get(SCOPE, "adls-tenant-id")

base = "fs.azure.account"
endpoint = f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"

spark.conf.set(f"{base}.auth.type.{ACCOUNT}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"{base}.oauth.provider.type.{ACCOUNT}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"{base}.oauth2.client.id.{ACCOUNT}.dfs.core.windows.net", client_id)
spark.conf.set(f"{base}.oauth2.client.secret.{ACCOUNT}.dfs.core.windows.net", client_secret)
spark.conf.set(f"{base}.oauth2.client.endpoint.{ACCOUNT}.dfs.core.windows.net", endpoint)

# 경로 정의
LOGS_PATH       = f"abfss://logs@datacopsadls.dfs.core.windows.net/gx_runs/date={today}/"
QUARANTINE_BASE = "abfss://quarantine@datacopsadls.dfs.core.windows.net"
GOLD_BASE       = "abfss://gold@datacopsadls.dfs.core.windows.net"

key = f"{base}.auth.type.{ACCOUNT}.dfs.core.windows.net"
print(f"[INFO] ADLS 인증 방식: {spark.conf.get(key)}")
print("[OK] 환경설정 완료")
print(f"[INFO] 로그 경로      : {LOGS_PATH}")
print(f"[INFO] Quarantine 경로: {QUARANTINE_BASE}")
print(f"[INFO] Gold 경로      : {GOLD_BASE}")

In [0]:
# ── 셀 2: GX 로그 읽기 + 파싱 ────────────────────────
# JSON 파일이 multi-line 형식이므로 multiLine 옵션 사용
df_logs_raw = (
    spark.read
    .option("multiLine", "true")
    .option("recursiveFileLookup", "true")
    .json(LOGS_PATH)
)
print(f"[OK] 로그 파일 로드: {df_logs_raw.count()}건")

df_logs = (
    df_logs_raw
    .filter(F.col("source").isNotNull())
    .withColumn("base_date",   F.to_date(F.col("date")))
    .withColumn("domain_name", F.upper(F.col("source")))
    .withColumn("window_start",
        F.date_trunc("hour", F.to_timestamp(F.col("run_id")))
    )
)

print(f"[OK] GX 로그 파싱 완료: {df_logs.count()}건")
df_logs.show(3, truncate=False)

In [0]:
# ── 셀 3: Quarantine 데이터 읽기 ──────────────────────
df_quarantine_raw = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet(f"{QUARANTINE_BASE}/*/date={today}")
)

total_q = df_quarantine_raw.count()
print(f"[OK] Quarantine 로드: {total_q}건")
print(f"[INFO] 컬럼: {df_quarantine_raw.columns}")

# _quarantine_reason 샘플 확인
df_quarantine_raw.select("_quarantine_reason").show(5, truncate=False)

In [0]:
# ── 셀 4: Quarantine 규칙명 추출 ──────────────────────
# _quarantine_reason = ",ERROR_USER_INVALID,ANOMALY_LENGTH_ZSCORE"
# → 콤마로 split해서 규칙명별로 행 분리

df_quarantine = (
    df_quarantine_raw
    # _quarantine_ts에서 날짜 추출
    .withColumn(
        "base_date",
        F.to_date(F.col("_quarantine_ts"))
    )
    # domain은 경로에서 못 읽으니 _platform 또는 고정값 사용
    .withColumn(
        "domain_name",
        F.upper(F.regexp_extract(F.input_file_name(), r"/quarantine/([^/]+)/", 1))
    )
    # 앞의 콤마 제거 후 split
    .withColumn(
        "reason_cleaned",
        F.regexp_replace(F.col("_quarantine_reason"), "^,", "")
    )
    # 콤마로 분리해서 규칙명별 행으로 펼치기
    .withColumn(
        "failed_rule_name",
        F.explode(F.split(F.col("reason_cleaned"), ","))
    )
    .filter(F.col("failed_rule_name") != "")
    .filter(F.col("base_date").isNotNull())
    .select("base_date", "domain_name", "failed_rule_name")
)

print(f"[OK] 규칙명 추출 완료")
df_quarantine.groupBy("failed_rule_name").count().orderBy(F.desc("count")).show(10, truncate=False)

In [0]:
# ── 셀 5: Gold 1 — 도메인별 통합 품질 매트릭스 ────────
# agg_global_data_quality_metrics
# PK: window_start + domain_name

df_gold1 = (
    df_logs
    .groupBy("window_start", "domain_name")
    .agg(
        F.sum(F.col("silver_rows") + F.col("quarantine_rows"))
         .alias("total_ingested_rows"),
        F.sum("silver_rows").alias("passed_rows"),
        F.sum("quarantine_rows").alias("quarantined_rows"),
        F.round(F.avg("quality_score"), 2).alias("avg_quality_score"),
        F.round(F.min("quality_score"), 2).alias("min_quality_score"),
        F.round(F.max("quality_score"), 2).alias("max_quality_score"),
        F.count("epoch_id").alias("total_batches"),
    )
    .withColumn(
        "data_purity_rate",
        F.round(
            F.col("passed_rows") /
            F.when(F.col("total_ingested_rows") == 0, 1)
             .otherwise(F.col("total_ingested_rows")) * 100,
            2
        )
    )
    .withColumn("_gold_created_at", F.current_timestamp())
    .orderBy("window_start", "domain_name")
)

gold1_path = f"{GOLD_BASE}/agg_global_data_quality_metrics"
df_gold1.write.mode("append").parquet(gold1_path)

print(f"[OK] Gold 1 저장 완료 | {df_gold1.count()}행")
df_gold1.show(truncate=False)

In [0]:
# ── 셀 6: Gold 2 — 도메인별 핵심 에러 규칙 랭킹 ────────
# agg_domain_error_top_rules
# PK: base_date + domain_name + failed_rule_name

# Quarantine에서 규칙별 위반 횟수 집계
df_violations = (
    df_quarantine
    .groupBy("base_date", "domain_name", "failed_rule_name")
    .agg(
        F.count("*").alias("violation_count")
    )
)

# error_rank 계산
window_rank = Window.partitionBy("domain_name", "base_date") \
                    .orderBy(F.desc("violation_count"))

# violation_trend 계산 (전일 대비)
window_lag = Window.partitionBy("domain_name", "failed_rule_name") \
                   .orderBy("base_date")

df_gold2 = (
    df_violations
    .withColumn("error_rank", F.dense_rank().over(window_rank))
    .withColumn(
        "prev_violation_count",
        F.lag("violation_count", 1).over(window_lag)
    )
    .withColumn(
        "violation_trend",
        F.when(F.col("prev_violation_count").isNull(), F.lit("STABLE"))
         .when(F.col("violation_count") > F.col("prev_violation_count"), F.lit("UP"))
         .when(F.col("violation_count") < F.col("prev_violation_count"), F.lit("DOWN"))
         .otherwise(F.lit("STABLE"))
    )
    .drop("prev_violation_count")
    .withColumn("_gold_created_at", F.current_timestamp())
    .orderBy("base_date", "domain_name", "error_rank")
)

gold2_path = f"{GOLD_BASE}/agg_domain_error_top_rules"
df_gold2.write.mode("append").parquet(gold2_path)

print(f"[OK] Gold 2 저장 완료 | {df_gold2.count()}행")
df_gold2.show(20, truncate=False)

In [0]:
# ── 셀 7: 전체 확인 ────────────────────────────────────
print("=" * 60)
print("Gold 컨테이너 적재 결과 최종 확인")
print("=" * 60)

df_check1 = spark.read.parquet(f"{GOLD_BASE}/agg_global_data_quality_metrics")
df_check2 = spark.read.parquet(f"{GOLD_BASE}/agg_domain_error_top_rules")

print(f"\n✅ Gold 1 — agg_global_data_quality_metrics")
print(f"   행 수  : {df_check1.count()}행")
print(f"   컬럼 수: {len(df_check1.columns)}개")
df_check1.show(5, truncate=False)

print(f"\n✅ Gold 2 — agg_domain_error_top_rules")
print(f"   행 수  : {df_check2.count()}행")
print(f"   컬럼 수: {len(df_check2.columns)}개")
df_check2.show(10, truncate=False)

print("\n🎉 Gold 레이어 완성!")

### Gold 3 — agg_llm_cost_efficiency_metrics
AI 비용 로그 기반 LLM 비용 효율 집계


In [0]:
# ── 셀 9: Gold 3 — agg_llm_cost_efficiency_metrics ────
AI_LOGS_PATH = "abfss://logs@datacopsadls.dfs.core.windows.net/ai_cost_logs/"

# 1) multi-line JSON 읽기 (.txt 확장자여도 json reader는 정상 동작)
df_ai_raw = (
    spark.read
    .option("multiLine", "true")
    .option("recursiveFileLookup", "true")
    .json(AI_LOGS_PATH)
)
print(f"[OK] AI 비용 로그 로드: {df_ai_raw.count()}건")

# 2) 필드 추출
df_ai_parsed = (
    df_ai_raw
    .select(
        F.col("window_start"),
        F.col("domain_name"),
        F.col("rule_version"),
        F.col("generated_rule_count").cast("int"),
        F.col("total_tokens").cast("int"),
        F.col("estimated_cost_usd"),
        F.col("stage1.cost_usd").alias("stage1_cost_usd"),
        F.col("stage2.cost_usd").alias("stage2_cost_usd"),
        F.lit(0.0).alias("avg_ai_latency_sec"),  # 원본 JSON에 해당 필드 없음
    )
    .filter(F.col("window_start").isNotNull())
)

print(f"[OK] AI 로그 파싱 완료: {df_ai_parsed.count()}건")

# 4) 집계: window_start(1시간 truncate) + domain_name(대문자)
df_gold3 = (
    df_ai_parsed
    .withColumn("window_start_hr", F.date_trunc("hour", F.to_timestamp("window_start")))
    .withColumn("domain_name_upper", F.upper(F.col("domain_name")))
    .groupBy("window_start_hr", "domain_name_upper")
    .agg(
        F.first("rule_version").alias("rule_version"),
        F.sum("generated_rule_count").alias("generated_rule_count"),
        F.sum("total_tokens").alias("total_tokens"),
        F.round(F.sum("estimated_cost_usd"), 2).alias("estimated_cost_usd"),
        F.round(F.sum("stage1_cost_usd"), 6).alias("stage1_cost_usd"),
        F.round(F.sum("stage2_cost_usd"), 6).alias("stage2_cost_usd"),
        F.round(F.avg("avg_ai_latency_sec"), 4).alias("avg_ai_latency_sec"),
    )
    .withColumnRenamed("window_start_hr", "window_start")
    .withColumnRenamed("domain_name_upper", "domain_name")
    .withColumn("_gold_created_at", F.current_timestamp())
    .orderBy("window_start", "domain_name")
)

# 5) 저장
gold3_path = f"{GOLD_BASE}/agg_llm_cost_efficiency_metrics"
df_gold3.write.mode("append").parquet(gold3_path)

# 6) 결과 출력
print(f"[OK] Gold 3 저장 완료 | {df_gold3.count()}행")
print(f"[INFO] 컬럼: {df_gold3.columns}")
df_gold3.show(3, truncate=False)

In [0]:
df_check3 = spark.read.parquet(
    "abfss://gold@datacopsadls.dfs.core.windows.net/agg_llm_cost_efficiency_metrics"
)
print(f"행 수: {df_check3.count()}행")
print(f"컬럼: {df_check3.columns}")
df_check3.show(3, truncate=False)

In [0]:
# ── 셀 10: Gold → PostgreSQL sync ──────────────────────
import psycopg2

SCOPE = "kv-sense-team4"

def get_pg_conn():
    return psycopg2.connect(
        host=dbutils.secrets.get(SCOPE, "db-host"),
        dbname=dbutils.secrets.get(SCOPE, "db-name"),
        user=dbutils.secrets.get(SCOPE, "db-user"),
        password=dbutils.secrets.get(SCOPE, "db-password")
    )

# ── 1. web_main_dashboard sync (Gold 1) ──
rows_gold1 = df_gold1.collect()
conn = get_pg_conn()
cur = conn.cursor()
for row in rows_gold1:
    cur.execute("""
        INSERT INTO web_main_dashboard
            (window_start, domain_name, total_ingested_rows, passed_rows,
             quarantined_rows, data_purity_rate, min_quality_score, max_quality_score)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (window_start, domain_name) DO UPDATE SET
            total_ingested_rows = EXCLUDED.total_ingested_rows,
            passed_rows         = EXCLUDED.passed_rows,
            quarantined_rows    = EXCLUDED.quarantined_rows,
            data_purity_rate    = EXCLUDED.data_purity_rate,
            min_quality_score   = EXCLUDED.min_quality_score,
            max_quality_score   = EXCLUDED.max_quality_score,
            synced_at           = NOW()
    """, (
        row["window_start"], row["domain_name"],
        row["total_ingested_rows"], row["passed_rows"],
        row["quarantined_rows"], row["data_purity_rate"],
        row["min_quality_score"], row["max_quality_score"]
    ))
conn.commit()
conn.close()
print(f"[OK] web_main_dashboard sync 완료: {len(rows_gold1)}행")

# ── 2. web_error_ranking sync (Gold 2) ──
rows_gold2 = df_gold2.filter(F.col("error_rank") <= 5).collect()
conn = get_pg_conn()
cur = conn.cursor()
for row in rows_gold2:
    cur.execute("""
        INSERT INTO web_error_ranking
            (base_date, domain_name, failed_rule_name,
             violation_count, error_rank, violation_trend)
        VALUES (%s, %s, %s, %s, %s, %s)
        ON CONFLICT (base_date, domain_name, failed_rule_name) DO UPDATE SET
            violation_count = EXCLUDED.violation_count,
            error_rank      = EXCLUDED.error_rank,
            violation_trend = EXCLUDED.violation_trend,
            synced_at       = NOW()
    """, (
        row["base_date"], row["domain_name"], row["failed_rule_name"],
        row["violation_count"], row["error_rank"], row["violation_trend"]
    ))
conn.commit()
conn.close()
print(f"[OK] web_error_ranking sync 완료: {len(rows_gold2)}행")

# ── 3. web_ai_cost_metrics sync (Gold 3) ──
rows_gold3 = df_gold3.collect()
conn = get_pg_conn()
cur = conn.cursor()
for row in rows_gold3:
    cur.execute("""
        INSERT INTO web_ai_cost_metrics
            (window_start, domain_name, rule_version, generated_rule_count,
             total_tokens, estimated_cost_usd, avg_ai_latency_sec)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (window_start, domain_name) DO UPDATE SET
            rule_version        = EXCLUDED.rule_version,
            generated_rule_count= EXCLUDED.generated_rule_count,
            total_tokens        = EXCLUDED.total_tokens,
            estimated_cost_usd  = EXCLUDED.estimated_cost_usd,
            avg_ai_latency_sec  = EXCLUDED.avg_ai_latency_sec,
            synced_at           = NOW()
    """, (
        row["window_start"], row["domain_name"], row["rule_version"],
        row["generated_rule_count"], row["total_tokens"],
        row["estimated_cost_usd"], row["avg_ai_latency_sec"]
    ))
conn.commit()
conn.close()
print(f"[OK] web_ai_cost_metrics sync 완료: {len(rows_gold3)}행")

print("\n✅ Gold → PostgreSQL sync 전체 완료")